# Module 7.5 — Graph RAG

Graph RAG augments vector search with a **knowledge graph** to handle:
- Multi-hop queries (A → B → C)
- Relationship-heavy questions
- Entity disambiguation

Stack: **Neo4j** + **LangChain Neo4j integration**

When Graph RAG > Vector RAG:
- 'Who are the co-authors of authors who cited Einstein?'
- 'Which drugs interact with Drug X that treat Disease Y?'

In [ ]:
# !pip install langchain-neo4j neo4j
# Requires Neo4j running locally or on AuraDB (free tier)

from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

# ── Connect to Neo4j ──────────────────────────────────────────────────────────
# graph = Neo4jGraph(
#     url="bolt://localhost:7687",
#     username="neo4j",
#     password="your-password"
# )
#
# ── Ingest data ───────────────────────────────────────────────────────────────
# graph.query("""
# MERGE (p1:Person {name: 'Albert Einstein'})
# MERGE (p2:Person {name: 'Niels Bohr'})
# MERGE (p3:Person {name: 'Werner Heisenberg'})
# MERGE (t1:Theory {name: 'Relativity'})
# MERGE (t2:Theory {name: 'Quantum Mechanics'})
# MERGE (p1)-[:DEVELOPED]->(t1)
# MERGE (p2)-[:CONTRIBUTED_TO]->(t2)
# MERGE (p3)-[:CONTRIBUTED_TO]->(t2)
# MERGE (p1)-[:DEBATED_WITH]->(p2)
# """)
#
# ── Natural language → Cypher chain ──────────────────────────────────────────
# llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# chain = GraphCypherQAChain.from_llm(llm, graph=graph, verbose=True)
# result = chain.invoke("Who developed the Theory of Relativity?")
# print(result["result"])

# Simulated demo (runs without Neo4j)
print("Graph RAG Demo (simulated — connect Neo4j for real queries)\n")

KNOWLEDGE_GRAPH = {
    "Albert Einstein": {"DEVELOPED": ["Relativity"], "DEBATED_WITH": ["Niels Bohr"]},
    "Niels Bohr":      {"CONTRIBUTED_TO": ["Quantum Mechanics"], "DEBATED_WITH": ["Albert Einstein"]},
    "Werner Heisenberg": {"CONTRIBUTED_TO": ["Quantum Mechanics"]},
}

def simple_graph_query(entity: str, relation: str) -> list:
    node = KNOWLEDGE_GRAPH.get(entity, {})
    return node.get(relation, [])

# Multi-hop: Who debated with people who developed Relativity?
relativity_devs = [e for e, rels in KNOWLEDGE_GRAPH.items() if "Relativity" in rels.get("DEVELOPED", [])]
print(f"Developers of Relativity: {relativity_devs}")
for dev in relativity_devs:
    debaters = simple_graph_query(dev, "DEBATED_WITH")
    print(f"  {dev} debated with: {debaters}")
